# sec-fundamental-tool -- Colab runner

順序：裝套件 -> 掛載 Drive -> 抓/更新程式碼(git) -> 設定信箱 -> 跑批次流程。

**第一次使用前**，把下面 `REPO_URL` 換成你自己在 GitHub 上的 repo 網址。

In [ ]:
# 1. 裝套件（每次開新 session 都要重新裝，Colab 不會記得上次裝過什麼）
!pip install -q pandas numpy openpyxl yfinance

In [ ]:
# 2. 掛載 Google Drive，輸出的 Excel/圖表會存到這裡，方便同學或自己其他電腦存取
from google.colab import drive
drive.mount('/content/drive')

# 你想把結果存到 Drive 哪裡，自己改這行路徑（建議用共用資料夾，同學也能看得到）
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/sec-fundamental-tool-outputs'

In [ ]:
# 3. 把程式碼從 GitHub 拉下來（第一次是 clone，之後重跑會自動改成 pull 最新版本）
REPO_URL = 'https://github.com/<your-username>/sec-fundamental-tool.git'  # 改成你自己的 repo 網址
PROJECT_DIR = '/content/sec-fundamental-tool'

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}

In [ ]:
# 4. 設定 SEC 聯絡信箱（不要寫死在程式碼裡，每次開新 session 在這邊輸入就好，不會被 commit 進 git）
import os
os.environ['SEC_CONTACT_EMAIL'] = 'angela741227@yahoo.com.tw'  # 改成你自己的信箱

In [ ]:
# 5. 跑批次流程（1-6），可以直接列想查的 ticker，也可以留空改成讀 config/tickers.csv
!python scripts/run_pipeline.py AAPL PG NVDA

In [ ]:
# 6. 拆股偵測（只偵測，不會自動改數字）
!python scripts/07_detect_stock_splits.py

# 看候選清單，人工確認後手動編輯 config/confirmed_splits.csv（直接在 Colab 左邊檔案總管點開來改）
import pandas as pd
print(pd.read_csv('outputs/06_quality_check/07_split_candidates.csv'))

In [ ]:
# 7. 確認過 config/confirmed_splits.csv 之後，套用拆股調整
!python scripts/08_apply_split_adjustments.py

In [ ]:
# 8. 把結果同步到 Google Drive，同學才看得到
!mkdir -p "{DRIVE_OUTPUT_DIR}"
!cp -r outputs/excel "{DRIVE_OUTPUT_DIR}/"
!cp -r outputs/charts "{DRIVE_OUTPUT_DIR}/" 2>/dev/null || true
print('Synced to', DRIVE_OUTPUT_DIR)